In [1]:
# --- environment guard -------------------------------------------------------
# These notebooks import the gateway, which needs the `gateway` extra (and `bench`
# for arm C's bi-encoder). Without them a later cell fails with a bare
# ModuleNotFoundError that reads like a code bug rather than a setup problem.
import importlib.util, sys

missing = [m for m in ("mcp", "sentence_transformers") if importlib.util.find_spec(m) is None]
if missing:
    print("Kernel is missing:", missing)
    print("Kernel python is:", sys.executable)
    print()
    print("Fix once, then select the 'mcp-gateway (.venv)' kernel:")
    print("    uv sync --extra gateway --extra bench --extra dev")
    print("    .venv/bin/python -m ipykernel install --user --name mcp-gateway")
    print()
    print("Or run headless with: uv run --with jupyter --with nbconvert "
          "--extra gateway --extra bench jupyter nbconvert --to notebook "
          "--execute --inplace <notebook>")
    raise SystemExit("wrong kernel")

print("kernel ok:", sys.executable)

kernel ok: /Users/nguyenvietkhoi/.cache/uv/builds-v0/.tmpnaJS8f/bin/python


# The gateway, stage by stage

**What this is for.** Prose kept collapsing two distinctions that matter: *shadow vs live*,
and *our selection vs Claude's tool search*. This notebook runs the real gateway at each
stage and shows what actually moves, so neither can stay ambiguous.

Every cell below runs against the live system — real MCP servers, the real catalog, the
real log format. Nothing here is illustrative.

**The one thing to hold onto:** the gateway sits on the **MCP wire**, between the client
and the upstream servers. It never sees the model call. That single fact explains most of
what follows — including why it can answer one of our two questions and not the other.

In [2]:
import json, os, subprocess, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

# Secrets come from .env beside gateway.json, exactly as the gateway itself loads them.
for line in (REPO / ".env").read_text().splitlines():
    line = line.strip()
    if line and not line.startswith("#") and "=" in line:
        k, _, v = line.partition("=")
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

print("repo:", REPO)

repo: /Users/nguyenvietkhoi/mcp-gateway-router


## Stage 1 — What the gateway knows before you type anything

This is the **entire** input at decision point A. There is no prompt at `tools/list`, so
whatever is captured here *is* the feature vector any point-A ranker gets.

Note where it comes from: the launcher shim exports the client's working directory
**before** `uv --directory` changes it. `os.getcwd()` would give the gateway's own repo,
and `$PWD` fails too because bash overwrites it at startup.

In [3]:
from mcp_gateway_router.gateway.environment import capture_environment

os.environ["MCP_GATEWAY_CLIENT_CWD"] = str(REPO)
env = capture_environment()
for k, v in sorted(env.items()):
    print(f"  {k:12s} {v}")

  branch       feat/gateway-data-plane
  cwd          /Users/nguyenvietkhoi/mcp-gateway-router
  hour         03
  project      mcp-gateway-router
  repo         mcp-gateway-router
  repo_path    /Users/nguyenvietkhoi/mcp-gateway-router
  weekday      Mon


## Stage 2 — The catalog, and why cost is not uniform

`harvest.py` connects to every upstream, aggregates their tools, and writes the artifact
the gateway reads at request time. Measurement is **control plane**: the data plane never
calls out to size a schema.

The spread below is why `fill_budget` is a knapsack rather than a top-K. Under a flat
cost-per-tool the two are the same operation.

In [4]:
from mcp_gateway_router.harvest import catalog_from_json
import statistics, collections

payload = json.loads((REPO / "results/catalog.json").read_text())
catalog = catalog_from_json(payload)

def size(t):
    return len(t.name) + len(t.description) + len(json.dumps(t.input_schema, separators=(",", ":")))

sizes = sorted(size(t) for t in catalog)
by_server = collections.Counter(t.server_id for t in catalog)

print(f"{len(catalog)} tools:", dict(by_server))
print(f"  min {sizes[0]:,}  median {int(statistics.median(sizes)):,}  max {sizes[-1]:,} chars")
print(f"  spread {sizes[-1]/sizes[0]:.0f}x")
print()
print("  bytes by server (cost concentrates, and not where tool COUNT suggests):")
tot = collections.Counter()
for t in catalog:
    tot[t.server_id] += size(t)
grand = sum(tot.values())
for s, n in tot.most_common():
    print(f"    {s:11s} {by_server[s]:>3} tools  {n:>7,} chars  {100*n/grand:>5.1f}% of bytes")

95 tools: {'playwright': 24, 'notion': 24, 'github': 47}
  min 146  median 1,001  max 5,743 chars
  spread 39x

  bytes by server (cost concentrates, and not where tool COUNT suggests):
    notion       24 tools   73,554 chars   54.3% of bytes
    github       47 tools   47,100 chars   34.8% of bytes
    playwright   24 tools   14,860 chars   11.0% of bytes


## Stage 3 — Shadow vs live, run for real

Same binary, same upstreams, same prompt. **One config field differs.**

- **shadow** — advertise the whole catalog, and *log the selection we would have made*.
  Behaviour is identical to having no gateway, so the client's own tool search still does
  the cut.
- **live** — advertise the cut.

The helper below launches an actual gateway subprocess and asks it `tools/list`.

⚠️ The `env` argument is load-bearing. `env=None` on `StdioServerParameters` is **not**
"inherit" — the MCP SDK substitutes a sanitised default carrying only
`HOME/LOGNAME/PATH/SHELL/TERM/USER`. Omit it and the gateway silently serves its default
config while looking perfectly healthy.

In [5]:
import asyncio, logging
logging.basicConfig(stream=sys.stderr, level=logging.ERROR)

from mcp_gateway_router.gateway.config import UpstreamSpec
from mcp_gateway_router.gateway.stdio_session import StdioUpstreamSession

SHIM = str(REPO / "bin/mcp-gateway")

async def ask_gateway(config_name, task=None):
    env = {"MCP_GATEWAY_CONFIG": str(REPO / config_name)}
    if task:
        env["MCP_GATEWAY_TASK"] = task
    s = StdioUpstreamSession(UpstreamSpec(server_id="gw", command=SHIM, env=env))
    await s.start()
    try:
        return [t.name for t in await s.list_tools()]
    finally:
        await s.aclose()

# Top-level await: the Jupyter kernel already runs an event loop, so
# asyncio.run() would raise here.
shadow = await ask_gateway("gateway.json")
print(f"shadow advertises {len(shadow)} tools -> the client sees everything")
print("  first few (catalog order):", ", ".join(n.split("__", 1)[1] for n in shadow[:4]))

shadow advertises 95 tools -> the client sees everything
  first few (catalog order): browser_close, browser_resize, browser_console_messages, browser_handle_dialog


In [6]:
TASK = "which pull requests are open on this repository"
live = await ask_gateway("gateway.armC.json", task=TASK)

print(f"live advertises {len(live)} tools for: {TASK!r}")
for n in live:
    print("   ", n)

live advertises 10 tools for: 'which pull requests are open on this repository'
    github__create_pull_request
    github__list_pull_requests
    github__pull_request_read
    github__update_pull_request
    github__merge_pull_request
    github__search_pull_requests
    github__update_pull_request_branch
    github__request_copilot_review
    github__get_copilot_job_status
    github__fork_repository


## Stage 4 — The decision record, read side by side

`exposed: []` used to be ambiguous — it read identically for *"shadow, the client received
everything"* and *"live, the selector chose nothing"*, which are opposite situations. A
reader hit exactly that and concluded the selector was broken.

The record now carries `mode` and `n_advertised`, so the two are distinguishable.

In [7]:
import glob

def latest_record(log_dir):
    files = sorted(glob.glob(str(REPO / log_dir / "*.jsonl")))
    if not files:
        return None
    for line in reversed(Path(files[-1]).read_text().splitlines()):
        r = json.loads(line)
        if r["kind"] == "decision":
            return r
    return None

rows = [("field", "shadow", "live (arm C)")]
a, b = latest_record("runs"), latest_record("runs/bench/C")
for f in ("mode", "n_candidates", "n_advertised", "selector_version", "decision_point"):
    rows.append((f, str(a and a.get(f)), str(b and b.get(f))))
rows.append(("exposed (selector chose)", str(a and len(a["exposed"])), str(b and len(b["exposed"]))))

w = [max(len(r[i]) for r in rows) for i in range(3)]
for i, r in enumerate(rows):
    print("  " + "  ".join(c.ljust(w[j]) for j, c in enumerate(r)))
    if i == 0:
        print("  " + "  ".join("-" * x for x in w))

  field                     shadow      live (arm C)
  ------------------------  ----------  ------------
  mode                      shadow      live        
  n_candidates              95          95          
  n_advertised              95          10          
  selector_version          static-set  semantic    
  decision_point            A           C           
  exposed (selector chose)  0           10          


### Reading that table

- **`decision_point: A`** in shadow — no prompt exists at `tools/list`, so `task` is `None`.
  Arm C shows **C** only because the harness injected the task via `MCP_GATEWAY_TASK`.
  That injection is what makes A-vs-C a comparison rather than a category error.
- **`n_advertised`** is what the client actually received. **`exposed`** is what the
  selector chose. In shadow they diverge on purpose: the selection is a *counterfactual*,
  recorded but never applied.

## Stage 5 — Where Claude's tool search sits, and why the log can't see it

```
  ┌─ Claude Code ──────────────────────────────────────┐
  │  1. tools/list ───────────────────────────────────────▶ gateway
  │  2. holds all 95 definitions locally                │
  │  3. builds the API request with schemas DEFERRED    │
  │  4. you prompt                                      │
  │  5. model emits ToolSearch                          │
  │  6. Claude Code answers it from its local copy      │
  │  7. model emits the real call ───────────────────────▶ gateway
  └─────────────────────────────────────────────────────┘
```

**Steps 2–6 never cross the MCP wire.** The cell below shows the consequence directly:
sessions log the resulting tool calls and never a search.

In [8]:
records = []
for f in glob.glob(str(REPO / "runs/*.jsonl")):
    records += [json.loads(l) for l in Path(f).read_text().splitlines() if l.strip()]

calls = [r for r in records if r["kind"] == "call"]
print(f"{len({r['session_id'] for r in records})} sessions, {len(calls)} tool calls logged")
print("any record mentioning a search:",
      any("search" in json.dumps(r).lower() and r["kind"] != "call" for r in records))
print()
print("what the gateway saw:")
for uid, n in collections.Counter(r["tool_uid"] for r in calls).most_common(6):
    print(f"   {uid:45s} x{n}")
print()
print("ToolSearch calls visible here: 0 — they are resolved inside the client.")

8 sessions, 7 tool calls logged
any record mentioning a search: False

what the gateway saw:
   notion/API-retrieve-page-markdown             x5
   github/search_code                            x1
   playwright/browser_navigate                   x1

ToolSearch calls visible here: 0 — they are resolved inside the client.


## Stage 6 — What the log becomes for the engine

The replay harness reconstructs sessions and hands a selector **only what preceded** the
session being predicted. History is passed at *construction*, so lookahead is impossible
rather than merely discouraged.

In [9]:
from mcp_gateway_router.replay.sessions import load_sessions

sessions = load_sessions(REPO / "runs")
print(f"{len(sessions)} sessions reconstructed\n")
for s in sessions[-3:]:
    print(f"  {s.session_id}")
    print(f"    opened   {s.opened_at.isoformat()}")
    print(f"    env      {json.dumps({k: v for k, v in s.environment.items() if k != 'repo_path'})}")
    print(f"    called   {[f'{a}/{b}' for a, b in s.called] or '(none)'}")

8 sessions reconstructed

  s-1786340260-6d79c52a
    opened   2026-08-10T05:37:40.943839+00:00
    env      {"branch": "feat/gateway-data-plane", "cwd": "/Users/nguyenvietkhoi/mcp-gateway-router", "hour": "01", "project": "mcp-gateway-router", "repo": "mcp-gateway-router", "weekday": "Mon"}
    called   (none)
  s-1786346004-b7b8948b
    opened   2026-08-10T07:13:24.877833+00:00
    env      {"branch": "feat/gateway-data-plane", "cwd": "/Users/nguyenvietkhoi/mcp-gateway-router", "hour": "03", "project": "mcp-gateway-router", "repo": "mcp-gateway-router", "weekday": "Mon"}
    called   (none)
  s-1786348091-f35cf99b
    opened   2026-08-10T07:48:12.215771+00:00
    env      {"branch": "feat/gateway-data-plane", "cwd": "/Users/nguyenvietkhoi/mcp-gateway-router", "hour": "03", "project": "mcp-gateway-router", "repo": "mcp-gateway-router", "weekday": "Mon"}
    called   (none)


## What this notebook establishes

| Question | Answer |
|---|---|
| What does the gateway intake at point A? | project · repo · branch · hour · weekday. No prompt. |
| What differs between shadow and live? | One config field. Shadow advertises everything and logs a counterfactual; live advertises the cut. |
| Can the gateway see Claude's tool search? | **No.** It resolves inside the client, never crossing the MCP wire. |
| What can this data answer? | **Q2** — can we predict a session's tools at open. |
| What can it *not* answer? | **Q1** — prefix cost and task success, which need the client. See `arms_walkthrough.ipynb`. |